# Why Do Models Fail on Rare Diseases?
## Notebook 3 — Looped Transformers with Adaptive Depth (CCS label space, MIMIC-IV)
University of Cambridge — Pietro Liò Group | Timothée Roina, 2026

**Method source.** Popescu, Sáez de Ocáriz Borde & Liò, *Adaptive Depth in Looped Transformers:
Diagnosing Learned Halting Gates and Trajectory Readouts* (arXiv:2607.20519, 2026).

**What changes relative to Notebook 2.** Only the three per-modality Transformers.
Embeddings, sum pooling, the shared readout `fc`, data, splits, loss and metrics are unchanged.

| | Notebook 2 (backbone) | Notebook 3 (looped) |
|---|---|---|
| Per-modality encoder | 1 PyHealth `TransformerBlock`, applied once | the **same** block applied $T$ times (weight-tied): $H_t = \mathcal{F}_\theta(H_{t-1})$ |
| Readout | `fc` on the final patient state | the **same** `fc` read out at every loop $t=1..T$ |
| Training loss | BCE at the output | per-depth BCE $\ell_t$ weighted by a depth prior or a learned exit distribution |
| Inference | always full depth | forced depth $t$, a post-hoc readout, or the learned gate decides when to stop |

**Three training objectives (from the paper).**
1. *Fixed-prior depth supervision*: $\mathcal{L} = \mathbb{E}\big[\sum_t \pi_t\,\ell_t\big]$, with $\pi$ uniform or truncated geometric ($\lambda=0.3$).
2. *Learned halting gate* (PonderNet-style): $e_t=\sigma(g_\phi(h_t))$, $q_t = e_t\prod_{j<t}(1-e_j)$, $q_T=\prod_{j<T}(1-e_j)$,
   $\mathcal{L} = \mathbb{E}\big[\sum_t q_t\,\ell_t + \beta\,\mathrm{KL}(q\,\|\,\pi)\big]$; inference exit $d_\alpha=\min\{t:\sum_{s\le t}q_s\ge\alpha\}$.
3. *Final-only* ($\pi=\delta_T$): a plain looped Transformer, i.e. the "depth probe" from the draft's Open Considerations.

**Post-hoc readouts (from the paper)**, recomputed on frozen trajectories, threshold calibrated on validation:
confidence (entropy, margin) and convergence (KL between consecutive predictions, hidden-state displacement).

**Adaptations to our setting — these are our choices, not the paper's.**
The paper studies single-label softmax tasks; ours is multi-label sigmoid over 572 CCS codes. Hence:
- $\ell_t$ = per-sample BCE averaged over labels (exactly Notebook 2's loss, evaluated at depth $t$).
- *Entropy* = mean binary entropy of the top-$m$ predicted codes ($m$ = median number of true codes, from Notebook 1).
- *Margin* = gap between the $m$-th and $(m{+}1)$-th predicted probabilities (stability of the top-$m$ cut).
- *KL* = mean Bernoulli KL between consecutive steps; *displacement* = $\|h_t-h_{t-1}\|/\|h_{t-1}\|$ on the concatenated patient state.
- Accuracy-based $D@X$ becomes: minimum average depth reaching $X\%$ of the full-depth validation sample-AUPRC.

**Questions this notebook answers for the paper.**
1. Does extra depth help the rarest quartile more than the common one (forced-exit curves per quartile)?
2. Do patients whose next visit contains a rare code need more loops (exit depth and oracle depth by group)?
3. As in the paper: do simple post-hoc readouts match or beat a learned gate on our task?

## 0. Installs

In [ ]:
# Run once in your environment (commented out so that batch execution never needs internet):
# %pip install pyhealth torch scikit-learn scipy numpy tqdm matplotlib --quiet

## 1. Imports, Paths, Configuration

In [ ]:
import json
import math
import os
import pickle
import random
import time
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import mannwhitneyu
from sklearn.metrics import average_precision_score
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

from pyhealth.models import TransformerLayer             # used only for the Notebook 2 equivalence check
from pyhealth.models.transformer import TransformerBlock # the block that TransformerLayer stacks

ROOT        = Path('..')
PROCESSED   = ROOT / 'processed'
RESULTS     = ROOT / 'results'
CHECKPOINTS = ROOT / 'checkpoints'
FIGURES     = ROOT / 'figures'
for d in [RESULTS, CHECKPOINTS, FIGURES]:
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
set_seed()

DEVICE = torch.device(
    'mps'  if torch.backends.mps.is_available() else
    'cuda' if torch.cuda.is_available() else
    'cpu'
)

# Same optimisation settings as Notebook 2
BATCH_SIZE = 64
MAX_EPOCHS = int(os.environ.get('LOOPED_MAX_EPOCHS', 50))   # env override for cluster jobs
PATIENCE   = 5
LR         = 1e-3
D_MODEL    = 256
N_HEADS    = 1
DROPOUT    = 0.5

RETRAIN = os.environ.get('LOOPED_RETRAIN', '0') == '1'   # False: reuse saved checkpoints when present
# PyHealth's TransformerBlock ends with Dropout(0.5) on its whole output. In a loop this would hit the
# recurrent state T times. Default (0): that dropout is moved to the readout branch only (see §5).
# Set to 1 to reproduce the first cluster runs (dropout on the recurrent state at every loop).
STREAM_DROPOUT = os.environ.get('LOOPED_STREAM_DROPOUT', '0') == '1'

print(f"Device: {DEVICE}")

## 2. Load Notebook 1 Artifacts

In [ ]:
with open(PROCESSED / 'meta_ccs.json') as f:
    meta = json.load(f)

code2idx = meta['code2idx']
D_out    = meta['N']
vocab    = meta['vocab']
m_median = meta['m_median']
m_90     = meta['m_90']
rare_idx_patient = meta['rare_idx']   # Notebook 1 rarity groups (patient-level counts)

def load_pkl(name):
    with open(PROCESSED / name, 'rb') as f:
        return pickle.load(f)

train_samples     = load_pkl('train_samples.pkl')
val_samples       = load_pkl('val_samples.pkl')
test_samples      = load_pkl('test_samples.pkl')
patient_sequences = load_pkl('patient_sequences.pkl')

print(f"CCS target vocabulary N = {D_out}")
print(f"Train / Val / Test      = {len(train_samples):,} / {len(val_samples):,} / {len(test_samples):,}")

## 3. Frequency Quartiles (identical to Notebook 2's MedCo-style evaluation)
Frequency = number of **training samples** in which a CCS code is a target. Test data never define groups.

In [ ]:
train_label_counts = Counter()
for s in train_samples:
    train_label_counts.update(set(s['label']))

train_freqs = np.array([train_label_counts[c] for c in vocab], dtype=float)
q25, q50, q75 = np.percentile(train_freqs, [25, 50, 75])

QUARTILES = {
    'Q1 0-25% (rarest)':  [i for i, f in enumerate(train_freqs) if f <= q25],
    'Q2 25-50%':          [i for i, f in enumerate(train_freqs) if q25 < f <= q50],
    'Q3 50-75%':          [i for i, f in enumerate(train_freqs) if q50 < f <= q75],
    'Q4 75-100% (common)':[i for i, f in enumerate(train_freqs) if f > q75],
}
RAREST, COMMON = 'Q1 0-25% (rarest)', 'Q4 75-100% (common)'
assert sum(len(v) for v in QUARTILES.values()) == D_out

print(f"Q25 = {q25:.0f} | Q50 = {q50:.0f} | Q75 = {q75:.0f}")
for k, v in QUARTILES.items():
    print(f"  {k:22s}: {len(v)} codes")

## 4. Vocabularies, Pre-computation and DataLoaders (copied from Notebook 2)

In [ ]:
dx_vocab_set, rx_vocab_set, px_vocab_set = set(), set(), set()
for pid, visits in patient_sequences.items():
    for _, dx, px, rx in visits:
        dx_vocab_set.update(dx); px_vocab_set.update(px); rx_vocab_set.update(rx)

dx2idx = {c: i + 1 for i, c in enumerate(sorted(dx_vocab_set))}
rx2idx = {c: i + 1 for i, c in enumerate(sorted(rx_vocab_set))}
px2idx = {c: i + 1 for i, c in enumerate(sorted(px_vocab_set))}
C_dx, C_rx, C_px = len(dx2idx) + 1, len(rx2idx) + 1, len(px2idx) + 1
print(f"C_dx = {C_dx} | C_rx = {C_rx} | C_px = {C_px}")


def pad_modality(seqs_batch, pad_idx=0):
    B = len(seqs_batch)
    T = max(len(s) for s in seqs_batch)
    L = max((len(v) for s in seqs_batch for v in s), default=1)
    code_idx  = torch.zeros(B, T, L, dtype=torch.long)
    code_mask = torch.zeros(B, T, L, dtype=torch.float)
    pad_mask  = torch.ones(B, T, dtype=torch.bool)
    lengths   = torch.zeros(B, dtype=torch.long)
    for b, seq in enumerate(seqs_batch):
        lengths[b] = len(seq) - 1
        for t, visit in enumerate(seq):
            pad_mask[b, t] = False
            for l, c in enumerate(visit):
                code_idx[b, t, l] = c
                if c != pad_idx:
                    code_mask[b, t, l] = 1.0
    return code_idx, code_mask, pad_mask, lengths


def collate_fn(batch):
    dx_seqs, rx_seqs, px_seqs, ys, hist_masks = zip(*batch)
    dx = pad_modality(dx_seqs); rx = pad_modality(rx_seqs); px = pad_modality(px_seqs)
    return (*dx, *rx, *px, torch.stack(ys), torch.stack(hist_masks))


def precompute_sequences(samples):
    processed = []
    for s in tqdm(samples, desc="Pre-computing"):
        visits = patient_sequences[s['patient_id']][:s['k']]
        dx_seq, rx_seq, px_seq, history_ccs = [], [], [], set()
        for _, dx, px, rx in visits:
            dx_seq.append([dx2idx[c] for c in dx if c in dx2idx] or [0])
            rx_seq.append([rx2idx[c] for c in rx if c in rx2idx] or [0])
            px_seq.append([px2idx[c] for c in px if c in px2idx] or [0])
            history_ccs.update(dx)
        y = torch.zeros(D_out)
        for c in s['label']:
            if c in code2idx:
                y[code2idx[c]] = 1.0
        hist_mask = torch.zeros(D_out, dtype=torch.bool)
        for c in history_ccs:
            if c in code2idx:
                hist_mask[code2idx[c]] = True
        processed.append((dx_seq, rx_seq, px_seq, y, hist_mask))
    return processed


class PrecomputedDataset(Dataset):
    def __init__(self, data): self.data = data
    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i]

train_proc, val_proc, test_proc = (precompute_sequences(s) for s in
                                   (train_samples, val_samples, test_samples))

def make_loader(data, shuffle):
    return DataLoader(PrecomputedDataset(data), batch_size=BATCH_SIZE, shuffle=shuffle,
                      collate_fn=collate_fn, num_workers=0)

train_loader = make_loader(train_proc, True)
val_loader   = make_loader(val_proc, False)
test_loader  = make_loader(test_proc, False)
print(f"Train / Val / Test batches: {len(train_loader):,} / {len(val_loader):,} / {len(test_loader):,}")

## 5. Looped Backbone

Per modality $m \in \{dx, rx, px\}$, with visit vectors $V^m$ from sum pooling (as in Notebook 2):

$$H^m_0 = V^m, \qquad H^m_t = \mathcal{F}^m_\theta\big(H^m_{t-1} \,[+\, V^m]\big), \qquad t = 1,\dots,T$$

where $\mathcal{F}^m_\theta$ is **one** PyHealth `TransformerBlock` (pre-norm attention + FFN), reused at every loop.
The bracketed input injection is off by default, matching the paper's $h_t = \mathcal{F}_\theta(h_{t-1})$.

At every loop we read out with the **same** head as Notebook 2:

$$h_t = \big[H^{dx}_t[:,0];\ H^{rx}_t[:,0];\ H^{px}_t[:,0]\big] \in \mathbb{R}^{3d}, \qquad z_t = W_{\text{out}}\,\mathrm{Dropout}(h_t) + b$$

**Pooling convention.** PyHealth's `TransformerLayer` returns `cls_emb = x[:, 0, :]`, the output at the
**first** visit position (the layer has no positional encoding and uses bidirectional attention). Notebook 2
therefore uses position 0, not the last visit as written in the paper draft. We keep position 0 so that $T=1$
reproduces Notebook 2 exactly (checked in §5b).

The optional gate is $g_\phi(h_t)$: a linear map $\mathbb{R}^{3d}\to\mathbb{R}$, or a two-layer MLP.

**Block output dropout.** PyHealth's `TransformerBlock.forward` ends with `return self.dropout(x)`, i.e.
Dropout($p=0.5$) on the **whole** output, residual stream included. With one pass (Notebook 2) this is applied
once. Inside a loop it would be applied to the recurrent state at every iteration, so after $T=4$ loops a
coordinate of $H_0$ survives on the direct path with probability $0.5^4 = 1/16$ during training. The first
cluster runs used that behaviour: deeper readouts were worse than depth 1, and `L4_uniform` collapsed at depths
3–4 after two epochs. By default (`STREAM_DROPOUT = False`) we therefore remove this dropout from the recurrent
path and apply it to the readout branch instead:
$z_t = W_{\text{out}}\,\mathrm{Dropout}(\mathrm{Dropout}_{\text{blk}}(h_t)) + b$, with $H_t$ passed on
undropped. The attention and FFN sublayer dropouts are unchanged. For $T=1$ the training distribution is
identical to Notebook 2 (same two dropouts on the readout), and in eval mode the two variants are identical
for every $T$.

In [ ]:
class LoopedBackbone(nn.Module):
    def __init__(self, C_dx, C_rx, C_px, D_out, d=256, n_heads=1, dropout=0.5,
                 T=4, input_injection=False, gate=None, stream_dropout=False):
        super().__init__()
        self.T = T
        self.input_injection = input_injection
        self.stream_dropout = stream_dropout

        self.emb_dx = nn.Embedding(C_dx, d, padding_idx=0)
        self.emb_rx = nn.Embedding(C_rx, d, padding_idx=0)
        self.emb_px = nn.Embedding(C_px, d, padding_idx=0)
        for emb in [self.emb_dx, self.emb_rx, self.emb_px]:
            nn.init.normal_(emb.weight, mean=0.0, std=0.01)
            emb.weight.data[0].fill_(0)

        # ONE block per modality, weight-tied across loops
        self.block_dx = TransformerBlock(d, n_heads, dropout)
        self.block_rx = TransformerBlock(d, n_heads, dropout)
        self.block_px = TransformerBlock(d, n_heads, dropout)
        if stream_dropout:                    # PyHealth default: block output dropout on the recurrent state
            self.readout_drop = nn.Identity()
        else:                                 # moved to the readout branch (no parameters: state_dict unchanged)
            for b in [self.block_dx, self.block_rx, self.block_px]:
                b.dropout = nn.Identity()
            self.readout_drop = nn.Dropout(dropout)

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(3 * d, D_out)          # shared readout across loops

        if gate == 'linear':
            self.gate = nn.Linear(3 * d, 1)
        elif gate == 'mlp':
            self.gate = nn.Sequential(nn.Linear(3 * d, d), nn.GELU(), nn.Linear(d, 1))
        else:
            self.gate = None

    @staticmethod
    def _visit_vectors(emb_layer, code_idx, code_mask):
        return (emb_layer(code_idx) * code_mask.unsqueeze(-1)).sum(dim=2)   # (B, V, d) sum pooling

    def forward(self, dx_idx, dx_mask, dx_pad, dx_len,
                      rx_idx, rx_mask, rx_pad, rx_len,
                      px_idx, px_mask, px_pad, px_len,
                      T=None, return_aux=False):
        T = T or self.T
        streams = []
        for emb_layer, block, idx, cmask, pad in [
                (self.emb_dx, self.block_dx, dx_idx, dx_mask, dx_pad),
                (self.emb_rx, self.block_rx, rx_idx, rx_mask, rx_pad),
                (self.emb_px, self.block_px, px_idx, px_mask, px_pad)]:
            v  = self._visit_vectors(emb_layer, idx, cmask)
            vm = (~pad).long()
            streams.append((block, v, torch.einsum('ab,ac->abc', vm, vm)))  # as in TransformerLayer

        hs = [v for _, v, _ in streams]
        logits, gate_logits, disp = [], [], []
        prev = None
        for t in range(T):
            hs = [block(h + v if (self.input_injection and t > 0) else h, am)
                  for (block, v, am), h in zip(streams, hs)]
            rep = torch.cat([h[:, 0, :] for h in hs], dim=1)          # PyHealth cls_emb convention
            logits.append(self.fc(self.dropout(self.readout_drop(rep))))
            if self.gate is not None:
                gate_logits.append(self.gate(rep).squeeze(-1))
            if return_aux:
                if prev is None:
                    disp.append(torch.full((rep.size(0),), float('inf'), device=rep.device))
                else:
                    disp.append((rep - prev).norm(dim=1) / (prev.norm(dim=1) + 1e-8))
                prev = rep

        out = {'logits': torch.stack(logits, dim=1)}                  # (B, T, N)
        if self.gate is not None:
            out['gate_logits'] = torch.stack(gate_logits, dim=1)       # (B, T)
        if return_aux:
            out['disp'] = torch.stack(disp, dim=1)                     # (B, T)
        return out


def to_device(batch):
    return [x.to(DEVICE) for x in batch]

batch = to_device(next(iter(train_loader)))
_m = LoopedBackbone(C_dx, C_rx, C_px, D_out, D_MODEL, N_HEADS, DROPOUT, T=4, gate='linear').to(DEVICE)
_o = _m(*batch[:12], return_aux=True)
print({k: tuple(v.shape) for k, v in _o.items()})
print(f"Parameters (T-independent): {sum(p.numel() for p in _m.parameters()):,}")
del _m, _o

### 5b. Sanity check — with $T=1$ the looped model is Notebook 2's backbone
We rebuild Notebook 2's `SharedBackbone`, copy its weights into a looped model with $T=1$, and compare
logits in eval mode. They must agree to floating-point precision.

In [ ]:
class SharedBackbone(nn.Module):          # verbatim architecture from Notebook 2
    def __init__(self, C_dx, C_rx, C_px, D_out, d=256, n_heads=1, n_layers=1, dropout=0.5):
        super().__init__()
        self.emb_dx = nn.Embedding(C_dx, d, padding_idx=0)
        self.emb_rx = nn.Embedding(C_rx, d, padding_idx=0)
        self.emb_px = nn.Embedding(C_px, d, padding_idx=0)
        self.transformer_dx = TransformerLayer(d, heads=n_heads, dropout=dropout, num_layers=n_layers)
        self.transformer_rx = TransformerLayer(d, heads=n_heads, dropout=dropout, num_layers=n_layers)
        self.transformer_px = TransformerLayer(d, heads=n_heads, dropout=dropout, num_layers=n_layers)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(3 * d, D_out)

    def _enc(self, emb_layer, transformer, code_idx, code_mask, pad_mask):
        v = (emb_layer(code_idx) * code_mask.unsqueeze(-1)).sum(dim=2)
        _, x = transformer(v, (~pad_mask).long())
        return x

    def forward(self, dx_idx, dx_mask, dx_pad, dx_len, rx_idx, rx_mask, rx_pad, rx_len,
                px_idx, px_mask, px_pad, px_len):
        x = torch.cat([self._enc(self.emb_dx, self.transformer_dx, dx_idx, dx_mask, dx_pad),
                       self._enc(self.emb_rx, self.transformer_rx, rx_idx, rx_mask, rx_pad),
                       self._enc(self.emb_px, self.transformer_px, px_idx, px_mask, px_pad)], dim=1)
        return self.fc(self.dropout(x))


def nb2_to_looped_state(sd):
    new = {}
    for k, v in sd.items():
        for m in ['dx', 'rx', 'px']:
            k = k.replace(f'transformer_{m}.transformer.0.', f'block_{m}.')
        new[k] = v
    return new

ref = SharedBackbone(C_dx, C_rx, C_px, D_out, D_MODEL, N_HEADS, 1, DROPOUT).to(DEVICE).eval()
loop1 = LoopedBackbone(C_dx, C_rx, C_px, D_out, D_MODEL, N_HEADS, DROPOUT, T=1).to(DEVICE).eval()
loop1.load_state_dict(nb2_to_looped_state(ref.state_dict()))
with torch.no_grad():
    diff = (ref(*batch[:12]) - loop1(*batch[:12])['logits'][:, 0]).abs().max().item()
print(f"max |logit difference| at T=1: {diff:.2e}")
assert diff < 1e-4, "Looped T=1 does not reproduce Notebook 2"

# If Notebook 2's checkpoint exists, it can initialise a looped model (a warm start; optional)
NB2_CKPT = CHECKPOINTS / 'backbone_ccs.pt'
print(f"Notebook 2 checkpoint found: {NB2_CKPT.exists()}")
del ref, loop1

## 6. Objectives

- Per-sample, per-depth loss: $\ell_t(x,y) = \frac{1}{N}\sum_{i} \mathrm{BCE}(z_{t,i}, y_i)$, which is Notebook 2's loss at depth $t$.
- Fixed priors: `final` ($\pi=\delta_T$), `uniform` ($\pi_t=1/T$), `geometric` ($\pi_t=\lambda(1-\lambda)^{t-1}$, $\pi_T=(1-\lambda)^{T-1}$).
- Learned gate: $\sum_t q_t\ell_t + \beta\,\mathrm{KL}(q\|\pi)$, with $q$ computed in log space for stability.

**Scale note on $\beta$.** Because $\ell_t$ averages BCE over 572 sparse labels, its magnitude is small
(on the order of $10^{-2}$), while the KL term is $O(1)$. A $\beta$ that works in the paper may be far too
strong here, so sweep it (`BETA_GRID` in §8).

In [ ]:
def depth_prior(T, kind, lam=0.3):
    if kind == 'final':
        p = [0.0] * (T - 1) + [1.0]
    elif kind == 'uniform':
        p = [1.0 / T] * T
    elif kind == 'geometric':
        p = [lam * (1 - lam) ** t for t in range(T - 1)] + [(1 - lam) ** (T - 1)]
    else:
        raise ValueError(kind)
    p = torch.tensor(p, dtype=torch.float32)
    assert abs(p.sum().item() - 1) < 1e-6
    return p


def exit_distribution(gate_logits):
    # q_t = e_t * prod_{j<t}(1 - e_j) for t < T ;  q_T = prod_{j<T}(1 - e_j). Returns (q, log q), shape (B, T).
    B, T = gate_logits.shape
    g = gate_logits[:, :-1]
    log_e, log_1me = F.logsigmoid(g), F.logsigmoid(-g)
    log_surv = torch.cat([gate_logits.new_zeros(B, 1), torch.cumsum(log_1me, dim=1)], dim=1)
    log_q = torch.cat([log_e + log_surv[:, :-1], log_surv[:, -1:]], dim=1)
    return log_q.exp(), log_q


def per_depth_bce(logits, y):                       # (B, T, N), (B, N) -> (B, T)
    return F.binary_cross_entropy_with_logits(
        logits, y.unsqueeze(1).expand_as(logits), reduction='none').mean(dim=-1)


def objective(out, y, cfg, prior):
    l = per_depth_bce(out['logits'], y)
    if cfg['objective'] == 'gate':
        q, log_q = exit_distribution(out['gate_logits'])
        kl = (q * (log_q - prior.log())).sum(dim=1)
        return ((q * l).sum(dim=1) + cfg['beta'] * kl).mean()
    return (l * prior).sum(dim=1).mean()


# Unit checks
_q, _ = exit_distribution(torch.randn(5, 4))
assert torch.allclose(_q.sum(1), torch.ones(5), atol=1e-6)
assert torch.allclose(exit_distribution(torch.randn(3, 1))[0], torch.ones(3, 1))
print("geometric(0.3), T=4:", depth_prior(4, 'geometric').numpy().round(3))

## 7. Metrics

**Fast exact sample-AUPRC.** Sample-AUPRC (Notebook 2's headline metric) is the mean, over samples with at least
one positive, of each sample's average precision across labels. So for **any** exit policy $d(x)$:

$$\text{sample-AUPRC}(d) = \frac{1}{|S|}\sum_{i\in S} \mathrm{AP}_{i,\,d(x_i)}$$

We therefore compute the per-sample AP table $\mathrm{AP}_{i,t}$ once per depth (overall and per quartile), and
every readout and threshold becomes a table lookup. The vectorised AP matches `sklearn` exactly when scores are
not tied (checked below).

In [ ]:
def per_sample_ap(scores, ys):
    # Average precision per row (labels ranked by score). NaN for rows with no positive.
    order = np.argsort(-scores, axis=1, kind='stable')
    y_sorted = np.take_along_axis(ys, order, axis=1)
    tp = np.cumsum(y_sorted, axis=1)
    prec = tp / np.arange(1, ys.shape[1] + 1)
    npos = ys.sum(axis=1)
    with np.errstate(invalid='ignore', divide='ignore'):
        ap = (prec * y_sorted).sum(axis=1) / npos
    ap[npos == 0] = np.nan
    return ap


def ap_tables(logits, ys):
    # Returns {'overall': (n, T), quartile_name: (n, T)}; NaN where the sample has no positive in that group.
    n, T, _ = logits.shape
    groups = {'overall': np.arange(D_out), **{k: np.array(v) for k, v in QUARTILES.items()}}
    return {g: np.stack([per_sample_ap(logits[:, t, cols], ys[:, cols]) for t in range(T)], axis=1)
            for g, cols in groups.items()}


def policy_score(table, d):
    # sample-AUPRC of exit policy d (0-based depth per sample) for one group's AP table
    v = table[np.arange(len(d)), d]
    return float(np.nanmean(v))


def macro_label_auprc(probs, ys, cols):
    aps = [average_precision_score(ys[:, j], probs[:, j]) for j in cols if ys[:, j].sum() > 0]
    return float(np.mean(aps)) if aps else float('nan')


def acc_at_m(probs, ys, m):
    top = np.argsort(-probs, axis=1)[:, :m]
    hits = np.take_along_axis(ys, top, axis=1).sum()
    return float(hits / ys.sum())


# Exactness check against sklearn on random untied scores
_r = np.random.default_rng(0)
_y = (_r.random((200, 50)) < 0.1).astype(float); _y[_y.sum(1) == 0, 0] = 1
_s = _r.random((200, 50))
assert abs(np.nanmean(per_sample_ap(_s, _y)) - average_precision_score(_y, _s, average='samples')) < 1e-10
print("per_sample_ap matches sklearn sample-AUPRC")

In [ ]:
@torch.no_grad()
def collect(model, loader):
    # One pass; returns per-depth logits (n, T, N), labels, gate logits, hidden displacement.
    model.eval()
    L, Y, G, D = [], [], [], []
    for batch in loader:
        batch = to_device(batch)
        out = model(*batch[:12], return_aux=True)
        L.append(out['logits'].float().cpu()); Y.append(batch[12].cpu()); D.append(out['disp'].cpu())
        if 'gate_logits' in out:
            G.append(out['gate_logits'].float().cpu())
    res = {'logits': torch.cat(L).numpy(), 'y': torch.cat(Y).numpy(), 'disp': torch.cat(D).numpy()}
    if G:
        res['gate_logits'] = torch.cat(G)
    return res


def sigmoid_np(x):
    return 1.0 / (1.0 + np.exp(-x))

## 8. Training

Early stopping uses validation sample-AUPRC at the **best depth** ($\max_t$), with Notebook 2's patience and
learning rate. For `final` this is in practice the depth-$T$ score. Each configuration is saved to
`../checkpoints/looped_<name>.pt` and reloaded when `RETRAIN = False`.

**Cost.** The visit embeddings are computed once; only the Transformer block is repeated. An epoch costs roughly
$T\times$ Notebook 2's encoder time. Start with `RUN = ['L1_final', 'L4_geometric']`: the first reproduces
Notebook 2, the second is the paper's best setting.

In [ ]:
T_MAX = 4          # paper: T=6 on synthetic tasks; Ouro checkpoints use up to 4 loops
BETA_GRID = [1e-4, 1e-3, 1e-2]

CONFIGS = {
    'L1_final':     dict(T=1,     objective='final',     prior='final'),               # = Notebook 2
    'L4_final':     dict(T=T_MAX, objective='final',     prior='final'),               # plain looped (depth probe)
    'L4_uniform':   dict(T=T_MAX, objective='fixed',     prior='uniform'),
    'L4_geometric': dict(T=T_MAX, objective='fixed',     prior='geometric', lam=0.3),  # paper's best setting
    **{f'L4_gate_b{b:g}': dict(T=T_MAX, objective='gate', prior='geometric', lam=0.3, beta=b, gate='linear')
       for b in BETA_GRID},
}
RUN = ['L1_final', 'L4_final', 'L4_uniform', 'L4_geometric', 'L4_gate_b0.001']
if os.environ.get('LOOPED_RUN'):                       # e.g. LOOPED_RUN=L4_final,L4_geometric
    RUN = [r.strip() for r in os.environ['LOOPED_RUN'].split(',') if r.strip()]
assert all(r in CONFIGS for r in RUN), f'Unknown config in RUN: {RUN}'
print('RUN =', RUN, '| MAX_EPOCHS =', MAX_EPOCHS, '| STREAM_DROPOUT =', STREAM_DROPOUT)

In [ ]:
def build_model(cfg):
    return LoopedBackbone(C_dx, C_rx, C_px, D_out, D_MODEL, N_HEADS, DROPOUT,
                          T=cfg['T'], input_injection=cfg.get('input_injection', False),
                          gate=cfg.get('gate'), stream_dropout=STREAM_DROPOUT).to(DEVICE)


def val_depth_scores(model):
    v = collect(model, val_loader)
    tab = ap_tables(v['logits'], v['y'])['overall']
    return np.nanmean(tab, axis=0)                       # sample-AUPRC at each depth


def train_config(name, cfg):
    ckpt = CHECKPOINTS / f'looped_{name}.pt'
    model = build_model(cfg)
    if ckpt.exists() and not RETRAIN:
        state = torch.load(ckpt, map_location=DEVICE)
        # checkpoints from the first runs have no flag: they used dropout on the recurrent state
        same_mode = state.get('stream_dropout', True) == STREAM_DROPOUT or cfg['T'] == 1   # T=1: equivalent
    if ckpt.exists() and not RETRAIN and not same_mode:
        print(f"[{name}] {ckpt.name} was trained with stream_dropout={state.get('stream_dropout', True)}"
              f" != {STREAM_DROPOUT}: ignored, retraining")
    elif ckpt.exists() and not RETRAIN:
        model.load_state_dict(state['model'])
        flag = '' if state.get('complete', True) else '  (WARNING: training was interrupted; best model so far)'
        print(f"[{name}] loaded {ckpt.name}{flag}")
        return model, state['history']

    set_seed()
    model = build_model(cfg)
    prior = depth_prior(cfg['T'], cfg['prior'], cfg.get('lam', 0.3)).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    best, best_state, wait, history = -1.0, None, 0, []

    for epoch in range(1, MAX_EPOCHS + 1):
        t0 = time.time()
        model.train()
        tot, n = torch.zeros((), device=DEVICE), 0
        for batch in train_loader:
            batch = to_device(batch)
            out = model(*batch[:12])
            loss = objective(out, batch[12], cfg, prior)
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tot += loss.detach() * batch[12].size(0); n += batch[12].size(0)

        per_t = val_depth_scores(model)
        score = float(np.nanmax(per_t))
        history.append({'epoch': epoch, 'train_loss': (tot / n).item(),
                        'val_sample_auprc_per_depth': per_t.tolist(), 'time_s': time.time() - t0})
        print(f"[{name}] epoch {epoch:>2} | {history[-1]['time_s']/60:.1f} min | "
              f"loss {history[-1]['train_loss']:.4f} | val sample-AUPRC by depth "
              + ' '.join(f'{x:.4f}' for x in per_t))
        if score > best:
            best, wait = score, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            # save at every improvement, so a job killed at its walltime still leaves the best model
            torch.save({'model': best_state, 'history': history, 'cfg': cfg, 'complete': False,
                        'stream_dropout': STREAM_DROPOUT}, ckpt)
        else:
            wait += 1
            if wait >= PATIENCE:
                print(f"[{name}] early stop at epoch {epoch} (best {best:.4f})")
                break

    model.load_state_dict(best_state)
    torch.save({'model': best_state, 'history': history, 'cfg': cfg, 'complete': True,
                'stream_dropout': STREAM_DROPOUT}, ckpt)
    return model, history


models, histories = {}, {}
for name in RUN:
    models[name], histories[name] = train_config(name, CONFIGS[name])

## 9. Collect Validation and Test Trajectories
Every result below is computed from these frozen trajectories: no further training.

In [ ]:
traj = {}
for name, model in models.items():
    traj[name] = {'val': collect(model, val_loader), 'test': collect(model, test_loader)}
    for split in ['val', 'test']:
        traj[name][split]['ap'] = ap_tables(traj[name][split]['logits'], traj[name][split]['y'])
print("Collected:", list(traj))

## 10. Forced-Exit Curves — does depth help the rarest quartile?
For each depth $t$ we evaluate the prediction at loop $t$ for every test sample. This is the paper's
"forced exit" per-depth quality curve, split by frequency quartile. The key diagnostic is
$\Delta_q = \text{AUPRC}_q(T) - \text{AUPRC}_q(1)$ for Q1 (rarest) versus Q4 (common).

In [ ]:
QCOLORS = {RAREST: '#0d366b', 'Q2 25-50%': '#1c5cab', 'Q3 50-75%': '#3987e5', COMMON: '#86b6ef'}
MARKERS = {RAREST: 'o', 'Q2 25-50%': 's', 'Q3 50-75%': '^', COMMON: 'D'}

forced = {}
for name in traj:
    te = traj[name]['test']
    probs = sigmoid_np(te['logits'])
    T = te['logits'].shape[1]
    rows = []
    for t in range(T):
        row = {'depth': t + 1, 'overall': float(np.nanmean(te['ap']['overall'][:, t]))}
        row.update({q: float(np.nanmean(te['ap'][q][:, t])) for q in QUARTILES})
        row['macro_label_rare_patientQ25'] = macro_label_auprc(probs[:, t], te['y'], rare_idx_patient)
        row[f'acc@{m_median}'] = acc_at_m(probs[:, t], te['y'], m_median)
        rows.append(row)
    forced[name] = rows

for name, rows in forced.items():
    print(f"\n{name}")
    print(f"{'depth':>5} {'overall':>8} " + ' '.join(f'{q[:2]:>8}' for q in QUARTILES)
          + f" {'macroRare':>10} {'acc@m':>7}")
    for r in rows:
        print(f"{r['depth']:>5} {r['overall']:8.4f} " + ' '.join(f'{r[q]:8.4f}' for q in QUARTILES)
              + f" {r['macro_label_rare_patientQ25']:10.4f} {r[f'acc@{m_median}']:7.4f}")
    if len(rows) > 1:
        d_rare = rows[-1][RAREST] - rows[0][RAREST]
        d_comm = rows[-1][COMMON] - rows[0][COMMON]
        print(f"  Δ(depth T − depth 1): rarest {d_rare:+.4f} | common {d_comm:+.4f}")

looped = [n for n in forced if len(forced[n]) > 1]
if looped:
    ncols = min(3, len(looped)); nrows = math.ceil(len(looped) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.4 * ncols, 3.4 * nrows), sharey=True, squeeze=False)
    for ax in axes.flat[len(looped):]:
        ax.axis('off')
    for ax, name in zip(axes.flat, looped):
        depths = [r['depth'] for r in forced[name]]
        for q in QUARTILES:
            ax.plot(depths, [r[q] for r in forced[name]], color=QCOLORS[q], marker=MARKERS[q],
                    markersize=7, linewidth=2, label=q)
        ax.set_title(name, fontsize=10)
        ax.set_xlabel('Forced exit depth t')
        ax.set_xticks(depths)
        ax.grid(alpha=0.25, linewidth=0.6)
        for s in ['top', 'right']:
            ax.spines[s].set_visible(False)
    for row in axes:
        row[0].set_ylabel('Test sample-AUPRC')
    handles, labels = axes.flat[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='lower center', ncol=4, frameon=False, fontsize=9)
    plt.tight_layout(rect=(0, 0.08 / nrows, 1, 1))
    plt.savefig(FIGURES / 'looped_forced_exit_by_quartile.png', dpi=150)
    plt.show()

## 11. Adaptive Exit — post-hoc readouts versus the learned gate
Each readout yields a scalar per sample and depth. The exit is the **first** depth at which the signal crosses
threshold $\tau$, with a forced exit at $T$. Thresholds come from a grid of validation quantiles. For the
learned gate we use the paper's native rule $d_\alpha$.

Reported per readout:
- the test Pareto curve (average depth vs sample-AUPRC, overall and rarest quartile);
- $D@X$: the smallest average depth whose **validation** score reaches $X\%$ of the full-depth validation
  score. We report the test depth and score at that validation-chosen threshold, so the test set never
  selects thresholds.

In [ ]:
EPS = 1e-7

def readout_signals(split):
    # {name: (signal (n,T), 'low'|'high')}. 'low': exit when signal <= tau; 'high': exit when signal >= tau.
    p = np.clip(sigmoid_np(split['logits']), EPS, 1 - EPS)
    n, T, N = p.shape
    top = -np.sort(-p, axis=2)
    topm = top[:, :, :m_median]
    ent = -(topm * np.log(topm) + (1 - topm) * np.log(1 - topm)).mean(axis=2)
    margin = top[:, :, m_median - 1] - top[:, :, m_median]
    kl = np.full((n, T), np.inf)
    if T > 1:
        a, b = p[:, 1:], p[:, :-1]
        kl[:, 1:] = (a * np.log(a / b) + (1 - a) * np.log((1 - a) / (1 - b))).mean(axis=2)
    return {'entropy@m': (ent, 'low'), 'margin@m': (margin, 'high'),
            'KL(t‖t-1)': (kl, 'low'), 'hidden displacement': (split['disp'], 'low')}


def first_exit(signal, direction, tau):
    ok = signal <= tau if direction == 'low' else signal >= tau
    ok = ok.copy(); ok[:, -1] = True
    return ok.argmax(axis=1)


def gate_exit(gate_logits, alpha):
    q, _ = exit_distribution(gate_logits)
    cdf = torch.cumsum(q, dim=1).numpy()
    ok = cdf >= alpha; ok[:, -1] = True
    return ok.argmax(axis=1)


def operating_points(name):
    val, te = traj[name]['val'], traj[name]['test']
    T = val['logits'].shape[1]
    policies = {}
    sv, st = readout_signals(val), readout_signals(te)
    for r, (sig_v, direction) in sv.items():
        finite = sig_v[np.isfinite(sig_v)]
        taus = np.unique(np.quantile(finite, np.linspace(0, 1, 41))) if finite.size else []
        policies[r] = [(tau, first_exit(sig_v, direction, tau), first_exit(st[r][0], direction, tau))
                       for tau in taus]
    if 'gate_logits' in val:
        policies['learned gate (d_α)'] = [(a, gate_exit(val['gate_logits'], a), gate_exit(te['gate_logits'], a))
                                          for a in list(np.linspace(0.05, 0.95, 19)) + [0.99]]
    policies['fixed depth'] = [(t, np.full(len(val['y']), t), np.full(len(te['y']), t)) for t in range(T)]

    pts = {}
    for r, lst in policies.items():
        pts[r] = [{'param': float(p),
                   'val_depth': float(dv.mean() + 1), 'test_depth': float(dt.mean() + 1),
                   'val_overall': policy_score(val['ap']['overall'], dv),
                   'test_overall': policy_score(te['ap']['overall'], dt),
                   'val_rarest': policy_score(val['ap'][RAREST], dv),
                   'test_rarest': policy_score(te['ap'][RAREST], dt),
                   'exit_test': dt} for p, dv, dt in lst]
    return pts


def d_at_x(pts, ref_val, x, key='overall'):
    ok = [p for p in pts if p[f'val_{key}'] >= x * ref_val]
    if not ok:
        return None
    best = min(ok, key=lambda p: p['val_depth'])
    return best


adaptive = {name: operating_points(name) for name in traj if traj[name]['val']['logits'].shape[1] > 1}

for name, pts in adaptive.items():
    val = traj[name]['val']
    ref = {k: float(np.nanmean(val['ap'][g][:, -1])) for k, g in [('overall', 'overall'), ('rarest', RAREST)]}
    print(f"\n{name} — full-depth val sample-AUPRC: overall {ref['overall']:.4f} | rarest {ref['rarest']:.4f}")
    print(f"{'readout':22s} {'target':>8} {'X':>4} {'test depth':>10} {'test score':>10}")
    for r, lst in pts.items():
        for key in ['overall', 'rarest']:
            for x in [0.95, 0.98, 0.99]:
                b = d_at_x(lst, ref[key], x, key)
                if b is None:
                    print(f"{r:22s} {key:>8} {int(x*100):>4} {'N/A':>10} {'N/A':>10}")
                else:
                    print(f"{r:22s} {key:>8} {int(x*100):>4} {b['test_depth']:10.2f} {b[f'test_{key}']:10.4f}")

In [ ]:
# Colour follows the readout, so it is identical across configurations
RSTYLE = {'entropy@m': ('#2a78d6', 'o'), 'margin@m': ('#eb6834', 's'), 'KL(t‖t-1)': ('#1baf7a', '^'),
          'hidden displacement': ('#eda100', 'D'), 'learned gate (d_α)': ('#e87ba4', 'v'),
          'fixed depth': ('#555555', 'x')}

for name, pts in adaptive.items():
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
    for ax, key, title in [(axes[0], 'overall', 'All codes'), (axes[1], 'rarest', 'Rarest quartile (Q1)')]:
        for r, lst in pts.items():
            lst = sorted(lst, key=lambda p: p['test_depth'])
            color, marker = RSTYLE[r]
            ax.plot([p['test_depth'] for p in lst], [p[f'test_{key}'] for p in lst],
                    color=color, marker=marker, markersize=5,
                    linewidth=2 if r != 'fixed depth' else 1.2,
                    linestyle='-' if r != 'fixed depth' else '--', label=r)
        ax.set_title(f'{name} — {title}', fontsize=10)
        ax.set_xlabel('Average exit depth (test)')
        ax.set_ylabel('Test sample-AUPRC')
        ax.grid(alpha=0.25, linewidth=0.6)
        for s in ['top', 'right']:
            ax.spines[s].set_visible(False)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='lower center', ncol=len(labels), frameon=False, fontsize=9)
    plt.tight_layout(rect=(0, 0.1, 1, 1))
    plt.savefig(FIGURES / f'looped_pareto_{name}.png', dpi=150)
    plt.show()

## 12. Rare-Disease Diagnostics

1. **Exit depth by group.** At a mid operating point (the readout threshold closest to the halfway average
   depth on validation), do test samples whose next visit contains a rarest-quartile code exit later?
   Tested with a two-sided Mann–Whitney U test.
2. **Oracle depth.** For each sample, the depth with the highest AP (ties go to the shallowest). If
   rare-target samples have a deeper oracle depth, extra computation is genuinely useful for them, whatever
   any readout manages to exploit. The oracle sample-AUPRC is an upper bound for any exit policy on this
   trajectory.

Labels are used only to **define the groups** for analysis, never to choose exits.

In [ ]:
def mwu_p(a, b):
    return float(mannwhitneyu(a, b, alternative='two-sided').pvalue) if len(a) and len(b) else float('nan')

diagnostics = {}
for name in adaptive:
    te = traj[name]['test']
    T = te['logits'].shape[1]
    has_rare = ~np.isnan(te['ap'][RAREST][:, 0])            # ≥1 rarest-quartile code in next visit
    oracle = np.nanargmax(np.nan_to_num(te['ap']['overall'], nan=-1), axis=1)
    keep = ~np.isnan(te['ap']['overall'][:, 0])

    res = {'n_rare_target': int(has_rare.sum()), 'n_other': int((~has_rare & keep).sum()),
           'oracle_depth_rare': float(oracle[has_rare].mean() + 1),
           'oracle_depth_other': float(oracle[~has_rare & keep].mean() + 1),
           'oracle_sample_auprc': policy_score(te['ap']['overall'], oracle),
           'full_depth_sample_auprc': policy_score(te['ap']['overall'], np.full(len(oracle), T - 1))}
    res['oracle_mwu_p'] = mwu_p(oracle[has_rare], oracle[~has_rare & keep])

    target = 1 + (T - 1) / 2
    res['exit_by_readout'] = {}
    for r, lst in adaptive[name].items():
        if r == 'fixed depth':
            continue
        p = min(lst, key=lambda p: abs(p['val_depth'] - target))
        d = p['exit_test']
        res['exit_by_readout'][r] = {'avg_depth': p['test_depth'],
                                     'depth_rare': float(d[has_rare].mean() + 1),
                                     'depth_other': float(d[~has_rare & keep].mean() + 1),
                                     'mwu_p': mwu_p(d[has_rare], d[~has_rare & keep])}
    diagnostics[name] = res

    print(f"\n{name}: {res['n_rare_target']:,} test samples with a rarest-quartile target, {res['n_other']:,} without")
    print(f"  oracle depth   rare {res['oracle_depth_rare']:.2f} vs other {res['oracle_depth_other']:.2f} "
          f"(MWU p = {res['oracle_mwu_p']:.2g})")
    print(f"  oracle sample-AUPRC {res['oracle_sample_auprc']:.4f} vs full depth {res['full_depth_sample_auprc']:.4f}")
    for r, v in res['exit_by_readout'].items():
        print(f"  {r:22s} avg {v['avg_depth']:.2f} | rare {v['depth_rare']:.2f} vs other {v['depth_other']:.2f} "
              f"(MWU p = {v['mwu_p']:.2g})")

## 13. Save Results

In [ ]:
def strip(pts):
    return {r: [{k: v for k, v in p.items() if k != 'exit_test'} for p in lst] for r, lst in pts.items()}

out = {
    'method': 'Looped Transformer, adaptive depth (Popescu, Saez de Ocariz Borde & Lio, arXiv:2607.20519)',
    'configs': {n: CONFIGS[n] for n in traj},
    'stream_dropout': STREAM_DROPOUT,
    'quartile_boundaries': {'q25': float(q25), 'q50': float(q50), 'q75': float(q75)},
    'forced_exit_test': forced,
    'adaptive_operating_points': {n: strip(p) for n, p in adaptive.items()},
    'rare_diagnostics': diagnostics,
    'histories': histories,
}
with open(RESULTS / 'looped_transformer_ccs_results.json', 'w') as f:
    json.dump(out, f, indent=2)
print("Saved → results/looped_transformer_ccs_results.json")

nb2 = RESULTS / 'backbone_ccs_medco_four_quartiles.json'
if nb2.exists():
    ref = json.load(open(nb2))
    print(f"\nNotebook 2 test sample-AUPRC: overall {ref['overall_sample_auprc']:.4f} | "
          f"rarest {ref['quartiles']['0-25% (rarest)']['sample_auprc']:.4f}")
print("\nDepth chosen on VALIDATION sample-AUPRC, scores on TEST:")
for name, rows in forced.items():
    t_best = int(np.nanargmax(np.nanmean(traj[name]['val']['ap']['overall'], axis=0)))
    r = rows[t_best]
    print(f"{name:16s} depth {r['depth']} | overall {r['overall']:.4f} | rarest {r[RAREST]:.4f} | "
          f"common {r[COMMON]:.4f} | gap {r[COMMON] - r[RAREST]:.4f}")

## 14. Reading the Results

| Observation | Interpretation for the paper |
|---|---|
| `L1_final` ≈ Notebook 2 | The pipeline is unchanged; any later difference is due to looping. |
| Q1 rises with depth while Q4 stays flat (§10) | Rare codes are partly **depth-limited**: one pass cannot extract their signal. |
| Q1 flat with depth | Depth is ruled out; this sharpens the attribution towards embeddings (Hypothesis A) or the readout (Hypothesis C). |
| Rare-target samples have a deeper oracle depth (§12) | Adaptive depth is worth pursuing specifically for rare diseases. |
| Post-hoc readouts ≥ learned gate (§11) | Replicates the paper on EHR data: the problem lies in the trajectory, not in gate capacity. |

**Caveats.** Results come from a single seed: re-run the configurations that matter with 3 seeds before drawing
conclusions. The Q1 test group is small (Notebook 2: 702 test samples), so differences below about 0.01 are
probably noise.